## Ficha: Orphanidou et al. (2015)

* **Referência / link / data de leitura:** Orphanidou, C. et al. (2015). *Signal-Quality Indices for the Electrocardiogram and Photoplethysmogram: Derivation and Applications to Wireless Monitoring*. IEEE Journal of Biomedical and Health Informatics, 19(3), 832–838. DOI: `10.1109/JBHI.2014.2338351`. Data de leitura: 20/11/2026.

* **Problema** (qual pergunta científica o artigo responde?): Como identificar automaticamente segmentos de ECG e PPG coletados por sensores vestíveis em pacientes ambulatoriais que sejam confiáveis ("good") para extrair frequência cardíaca (HR) — reduzindo alarmes falsos e fadiga de alarmes — e como utilizar esse índice de qualidade (SQI) para economizar bateria (duty cycling) e melhorar a estimativa de frequência respiratória (RR) a partir do PPG.

* **Dados** (bases, sujeitos, modalidades, condições):
  * **PCinC (PhysioNet/CinC Challenge 2011):** 1.500 segmentos de ECG de 12 derivações a 500 Hz (16 bits), utilizando apenas a 1ª derivação.
  * **JR (John Radcliffe Hospital, Oxford):** estudo clínico com pacientes adultos de enfermarias médicas, cirúrgicas e geriátricas capazes de andar sem assistência (escore ambulatorial 1–5, média 4,5 e mediana 5). Inclui ECG, PPG e RIP de três monitores: Dyna-Vision DVM012S (ECG 100 Hz / 16 bits, PPG 75 Hz / 12 bits), Equivital EQ-02 (ECG 256 Hz, RIP 25,6 Hz, PPG via XPOD 3012LP a 75 Hz / 16 bits) e Nonin WristOx2 3150 (PPG a 75 Hz / 6 bits).
  * Para desenvolver o ECG SQI foram selecionados **1.500 segmentos de 10 s** (500 de PCinC, 500 de $JR_{D-ECG}$ e 500 de $JR_{E-ECG}$). Para o PPG SQI, **1.500 segmentos de 10 s** (750 de $JR_{E-PPG}$ e 750 de $JR_{N-PPG}$). As aplicações usaram os registros completos do JR (ex.: 18 pacientes no ECG — 123 h no Dyna-Vision e 63 h no Equivital; 7 pacientes e 63 h no $JR_{E-PPG}$ / $JR_{E-RESP}$; 12 pacientes e 131 h no $JR_{N-PPG}$).

* **Split** (aleatório, por sujeito, LOSO? risco de leakage?): Dos 1.500 segmentos de ECG, **1.000 foram escolhidos aleatoriamente** para treino/otimização do limiar de correlação via validação cruzada de 5 dobras (5-fold CV) e **500 ficaram para teste**. A divisão não explicita agrupamento estrito por indivíduo (subject-wise / LOSO) na amostragem aleatória dos 1.500 segmentos, o que representa risco potencial de *data leakage* entre sujeitos no desenvolvimento do limiar.

* **Método** (sinais, features e modelos que entram na inferência): Detecção de picos R com Hamilton e Tompkins (ou detector de 3 pontos com limiares empíricos no PPG) em janelas de 10 s, seguida de 3 regras fisiológicas sequenciais (HR em 40–180 bpm, intervalo RR máximo $\le 3\text{ s}$, razão $\text{RR}_{\max}/\text{RR}_{\min} < 2{,}2$) e casamento de template adaptativo (correlação média de cada complexo/pulso com o template médio centrado no pico usando janela igual à mediana do RR), gerando saída binária `"GOOD"` / `"BAD"`.

* **Qualidade** (como SQI, movimento, pressão ou falhas são tratados): O próprio algoritmo define uma SQI binária (`"good"` vs. `"bad"`) voltada à confiabilidade da HR em sinais com artefato de movimento. Na aplicação de RR via PPG, janelas de 60 s (avançando 10 s) foram estratificadas pela duração contínua de artefato ($\le 60\text{ s}$, $\le 40\text{ s}$, $\le 20\text{ s}$ e $0\text{ s}$ / sem artefato), avaliando o compromisso entre erro (MAE) e frequência de estimativas.

* **Métrica** (MAE, F1 etc.; qual a unidade de análise?):
  * **SQI (unidade: segmento de 10 s):** Sensibilidade (proporção de `"bad"` corretamente identificado como `"bad"`) e Especificidade (proporção de `"good"` corretamente identificado como `"good"`).
  * **Power-saving (unidade: janela não sobreposta de 5 min):** % de janelas de 5 min com ao menos um segmento `"good"` de 10 s obtido e % de tempo de gravação economizado.
  * **RR via PPG (unidade: janela de 60 s, avançando a cada 10 s):** Erro Absoluto Médio (MAE, em bpm) e desvio padrão vs. padrão-ouro (RIP), além da contagem de estimativas de RR válidas.

* **Limitação** (o que os autores admitem ou o protocolo não cobre): O desempenho depende do detector de picos utilizado (picos perdidos ou falsos positivos ocorrem mais no PPG, morfologicamente mais variável). O algoritmo provavelmente **não funciona bem em pacientes com fibrilação atrial (FA)** devido à irregularidade do ritmo/intervalos RR. Além disso, o duty cycling assume 1 HR confiável a cada 5 min (risco de perder deterioração rápida se não houver gatilho de alta frequência), avaliado retrospectivamente em pacientes hospitalares ambulatoriais.

* **Para nosso artigo** (o que reutilizar conceitualmente): Empregar janelas de 10 s para o SQI de ECG e extrair features `ecg_` quantitativas (regras de viabilidade de RR, correlação com template adaptativo e proporção de trechos válidos) no pipeline reprodutível do BioSense-AI para filtrar/avaliar a confiabilidade do sinal antes de modelagem de alterações cardiovasculares e predição de risco.

* **O que este artigo impede que reivindiquemos como novidade:** Propor o uso de regras fisiológicas de intervalos RR (40–180 bpm, gap $\le 3\text{ s}$, razão max/min $< 2{,}2$) combinadas à correlação com template adaptativo de batimentos em janelas de 10 s para classificar qualidade de ECG em sinais ambulatoriais/vestíveis.

* **Qual lacuna dele é explorada pelo nosso trabalho:** O artigo foca na classificação binária (`"good"`/`"bad"`) para HR em janelas de 10 s, duty cycling a cada 5 min e RR via PPG, sem integrar essas métricas como features contínuas de qualidade em modelos de detecção de alterações cardiovasculares e risco, sem avaliar validação estritamente por indivíduo (subject-wise/LOSO) para evitar leakage, e admitindo fragilidade em ritmos irregulares (como FA) e ausência de análises de interpretabilidade, robustez e reprodutibilidade de pipeline.

## Ficha: Rathnayake et al. (2026)

* **Referência / link / data de leitura:** Rathnayake, C., Chen, W., Jayawickrama, S., & Lai, D. (2026). *Physiological Monitoring Applications of Wearable Multimodal Fusion Systems Based on ECG and PPG: A Comprehensive Review*. Sensors, 26(11), 3477. DOI: `10.3390/s26113477` / PMC13259056. Data de leitura: 20/11/2026.

* **Problema** (qual pergunta científica o artigo responde?): Como sistemas vestíveis de fusão multimodal baseados em ECG e PPG são projetados, sincronizados e aplicados no monitoramento fisiológico (pressão arterial sem manguito, estresse, HR e HRV), quais estratégias de fusão são usadas e quais gargalos técnicos e lacunas ainda limitam sua confiabilidade e adoção clínica?

* **Dados** (bases, sujeitos, modalidades, condições): Revisão narrativa estruturada (~300 artigos triados em bases como PubMed, IEEE Xplore, Scopus e Web of Science, com foco recente nos últimos 5 anos e marcos dos últimos 10 anos, excluindo sistemas hospitalares de beira de leito não vestíveis e trabalhos puramente teóricos). Abrange estudos com ECG (1 a 12 derivações, patches, cintas, roupas inteligentes, relógios, Holter, MCT) e PPG (pulso, dedo, orelha, anel, testa, braço, tórax), tanto em coletas proprietárias/vestíveis quanto em bases públicas/clínicas como MIMIC (I/II/III), VitalDB, MIT-BIH, PulseDB, WESAD, CLAS, BIDMC e EarSet.

* **Split** (aleatório, por sujeito, LOSO? risco de leakage?): Por ser uma revisão de literatura, não há um *split* único próprio; os estudos revisados variam entre *train-test split*, *k-fold CV*, *leave-one-subject-out* / *leave-one-person-out* e validação *cross-dataset* ou *cross-sensor*, sendo que muitos trabalhos ainda sofrem com amostras pequenas, cenários de laboratório e risco de generalização limitada quando não validam rigorosamente por sujeito ou em condições ambulatoriais reais.

* **Método** (sinais, features e modelos que entram na inferência): Sintetiza pipelines com aquisição de ECG e PPG, pré-processamento (filtragem passa-banda, remoção de *baseline wander*, *wavelet*, EMD/VMD/SSA, ICA/PCA, *denoising* por DL), janelamento (curto de 2–10 s para ECG/arritmias/HR; médio de 10–30 s para PTT/fusão; longo de 30 s–5 min para HRV/estresse), extração de *features* (HRV/PRV, morfologia de QRS/pulso, PTT/PAT, espectrais/wavelet) e modelos de ML/DL (RF, SVM, XGBoost, *boosting*, ANN, CNN, LSTM/Bi-LSTM, ResUNet). Classifica a fusão em **feature-level** (71,43% dos estudos revisados), **signal-level** (21,43%) e **decision-level** (7,14%).

* **Qualidade** (como SQI, movimento, pressão ou falhas são tratados): Aponta que artefatos de movimento vêm de variação de pressão de contato, atividade muscular, deformação da pele e deslocamento do sensor, somados a *baseline wander*, interferência de rede, luz ambiente e variabilidade de perfusão. Trata a qualidade por meio de filtragem/decomposição, *denoising* com IMU ou ML/DL, imputação ou reconstrução de dados faltantes, rejeição de segmentos ruins, ponderação por confiança (*confidence weighting*) e uso de **SQI** para excluir trechos contaminados antes da fusão ou modelagem.

* **Métrica** (MAE, F1 etc.; qual a unidade de análise?): Varía conforme a aplicação nos estudos revisados: **MAE** e desvio padrão (em mmHg, frequentemente avaliados frente a protocolos como AAMI/IEEE/BHS) na estimativa de PA sem manguito; **F1-score**, **acurácia**, **AUC** e **RMSE/AAE** (em bpm ou ms) na detecção de estresse, arritmias (como FA) e monitoramento de HR/HRV; as unidades de análise vão de batimento/intervalo a janelas temporais fixas (ex.: 10 s, 30 s ou minutos) e sujeitos/registros completos.

* **Limitação** (o que os autores admitem ou o protocolo não cobre): Revisão narrativa (não metanálise quantitativa), sem implementação própria nem benchmark experimental unificado; muitos estudos revisados têm amostras pequenas, validação apenas em laboratório ou em bases de UTI/clínicas (como MIMIC/VitalDB) em vez de vestíveis de uso diário, forte sensibilidade a artefatos de movimento e contato cutâneo, dependência de calibração, exigência computacional/energética em dispositivos vestíveis e ausência de padronização entre pipelines.

* **Para nosso artigo** (o que reutilizar conceitualmente): Reutilizar a organização do pipeline (pré-processamento, janelamento adequado à tarefa, tratamento de segmentos corrompidos via SQI/rejeição/ponderação e validação rigorosa) e posicionar a avaliação da qualidade do ECG por janela como base de confiabilidade do sinal. Lacunas apontadas pela revisão (7 no total):
  1. *Falta de estratégia padronizada ou ótima de fusão e uso de SQI para lidar com trechos ruidosos/ausentes:* muitos trabalhos não fazem fusão real ou combinam sinais sem critério uniforme de rejeição/ponderação por qualidade, o que reforça medir explicitamente a confiabilidade do ECG por janela antes de usar o sinal.
  2. *Alta sensibilidade a artefatos de movimento, contato e ruído em cenários ambulatoriais/vestíveis:* variações de contato, atividade física e ruído degradam picos e morfologia; quantificar a qualidade de cada janela de ECG ajuda a isolar trechos confiáveis de trechos contaminados.
  3. *Seleção de janela e compromisso entre resolução temporal, latência e estabilidade:* janelas curtas favorecem detecção rápida/arritmias e picos, enquanto janelas maiores estabilizam métricas estatísticas — avaliar a confiabilidade por janela permite saber quando uma janela tem informação válida suficiente.
  4. *Sincronização ECG–PPG majoritariamente estática em vez de adaptativa em tempo real:* o atraso fisiológico (PTT/PAT) varia dinamicamente com PA, rigidez vascular e movimento, mas a maioria dos modelos assume atraso fixo ou alinhamento simples.
  5. *Escassez de validação em larga escala no mundo real e de conjuntos padronizados:* muitos estudos usam bases clínicas/UTI ou coletas pequenas em laboratório, com pouca validação ambulatorial contínua e variabilidade demográfica (idade, sexo, IMC, tom de pele).
  6. *Restrições de energia, memória e computação em dispositivos vestíveis de borda:* modelos complexos e aquisição contínua esgotam bateria e recursos embarcados, exigindo soluções leves e eficientes.
  7. *Pouco uso de sensores integrados no mesmo local anatômico para aplicações práticas:* apesar de protótipos captarem ECG e PPG no mesmo ponto, ainda há pouca validação prática consolidada dessas configurações.

* **O que este artigo impede que reivindiquemos como novidade:** Reivindicar como novidade o fato de que artefatos de movimento e ruído degradam sinais vestíveis de ECG/PPG, que janelas temporais afetam latência e estabilidade das métricas, ou que a fusão multimodal (especialmente em nível de *features*) e o uso de SQI/filtragem são úteis para mitigar ruído e falhas em monitoramento fisiológico.

* **Qual lacuna dele é explorada pelo nosso trabalho:** A revisão mapeia amplamente sistemas de fusão ECG–PPG e seus desafios gerais, mas não aprofunda uma engenharia sistemática de *features* de confiabilidade do ECG por janela integrada a um protocolo experimental reprodutível com validação estrita por indivíduo, interpretabilidade e análise de robustez no escopo do BioSense-AI.

## Síntese comparativa: Orphanidou et al. (2015) vs. Rathnayake et al. (2026)

Enquanto **Orphanidou et al. (2015)** propõem um algoritmo concreto e de baixa complexidade para classificar janelas de 10 s de ECG e PPG em "good" ou "bad" — combinando regras fisiológicas de frequência cardíaca e intervalos RR (40–180 bpm, gap $\le 3\text{ s}$ e razão $\text{RR}_{\max}/\text{RR}_{\min} < 2{,}2$) com casamento de template adaptativo (limiar ótimo de 0,66 no ECG) e aplicações em *duty cycling* e frequência respiratória[cite: 3, 4, 5] —, **Rathnayake et al. (2026)** oferecem o panorama amplo de sistemas vestíveis de fusão ECG–PPG, organizando estratégias de fusão (dominadas por *feature-level fusion* em 71,43% dos estudos), métodos de sincronização, janelamento e tratamento de artefatos e falhas[cite: 8, 24, 29, 30]. Juntos, os dois textos fundamentam a medição explícita da confiabilidade do ECG por janela antes da modelagem: o primeiro fornece a base operacional e os limiares para avaliar segmentos de ECG[cite: 3, 4, 5], e o segundo evidencia as lacunas de qualidade, sincronização, generalização e padronização que motivam o protocolo reprodutível e validado por indivíduo do **BioSense-AI**[cite: 24, 38].